# Approach 12: DeepECG-SSL linear probing & fine-tuning

Linear probe on top of the frozen DeepECG-SSL encoder, then full fine-tune, for Chagas detection on CODE-15%. OOD eval on SaMi-Trop.

In [1]:
import os
import sys
import time
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    f1_score, roc_curve, precision_recall_curve,
)
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

sns.set_context("notebook")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 5)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("chagas")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"PyTorch {torch.__version__} | Device : {device}")

/Users/jwasieleski/Prywatne/jul/workspace/magisterka/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.10.0 | Device : mps


In [2]:
CFG = {
    "preprocessed_cache": "preprocessed_cache_brazil.h5",
    "val_fraction": 0.15,
    "test_fraction": 0.15,
    "random_seed": 42,
    "target_fs": 500,
    "target_duration_s": 10,
    
    "ssl_model_path": "DeepECG_Docker-main/ssl_pretrained_model/SSL_pretrained.pt",
    "fairseq_path": "DeepECG_Docker-main/fairseq-signals",
    "deepecg_path": "DeepECG_Docker-main",
    
    "epochs_linear": 10,
    "epochs_finetune": 20,
    "batch_size": 32,
    "lr_linear": 1e-3,
    "lr_finetune": 1e-5,
    "weight_decay": 1e-4,
    "num_workers": 0,
    
    "focal_alpha": 0.25,
    "focal_gamma": 2.0,
}

In [3]:
sys.path.append(os.path.abspath(CFG["deepecg_path"]))
sys.path.append(os.path.abspath(CFG["fairseq_path"]))

from fairseq_signals.utils import checkpoint_utils

In [4]:

cache_file = CFG["preprocessed_cache"]
assert Path(cache_file).exists(), f"Missing {cache_file} — run chagas_resnet_classifier.ipynb first."

with h5py.File(cache_file, "r") as f:
    n_total = f["labels"].shape[0]
    all_labels = f["labels"][:]
    all_exam_ids = f["exam_ids"][:] if "exam_ids" in f else np.arange(n_total)
    print(f"Cache — Samples: {n_total:,} | Chagas+ {100*all_labels.mean():.1f}%")

samitrop_dir = "sami-trop"
df_samitrop = pd.read_csv(os.path.join(samitrop_dir, "exams.csv"))
samitrop_id_set = set(df_samitrop["exam_id"].values)

is_samitrop = np.array([eid in samitrop_id_set for eid in all_exam_ids])
samitrop_indices = np.where(is_samitrop)[0]
code15_indices   = np.where(~is_samitrop)[0]

print(f"CODE-15% entries : {len(code15_indices):,}  ({100*all_labels[code15_indices].mean():.1f}% Chagas+)")
print(f"SaMi-Trop entries: {len(samitrop_indices):,}  (100% Chagas+, OOD only)")

from sklearn.model_selection import train_test_split as _tts

code15_labels = all_labels[code15_indices]

_tv_rel, _test_rel = _tts(
    np.arange(len(code15_indices)),
    test_size=0.15,
    stratify=code15_labels,
    random_state=CFG["random_seed"],
)

_tv_labels = code15_labels[_tv_rel]
_train_rel, _val_rel = _tts(
    np.arange(len(_tv_rel)),
    test_size=0.18,
    stratify=_tv_labels,
    random_state=CFG["random_seed"],
)

test_indices  = code15_indices[_test_rel]
_tv_indices   = code15_indices[_tv_rel]
train_indices = _tv_indices[_train_rel]
val_indices   = _tv_indices[_val_rel]

print()
print(f"{'='*62}")
print(f"  DATA SPLIT SUMMARY")
print(f"{'='*62}")
print(f"  Train : {len(train_indices):>6,}  ({100*all_labels[train_indices].mean():.1f}% Chagas+)")
print(f"  Val   : {len(val_indices):>6,}  ({100*all_labels[val_indices].mean():.1f}% Chagas+)")
print(f"  Test  : {len(test_indices):>6,}  ({100*all_labels[test_indices].mean():.1f}% Chagas+) [CODE-15% held-out]")
print(f"  OOD   : {len(samitrop_indices):>6,}  (100.0% Chagas+) [SaMi-Trop domain-shift]")
print(f"{'='*62}")

class CachedChagasDataset(Dataset):
    def __init__(self, cache_path, indices):
        self.cache_path = cache_path
        self.indices = np.sort(indices)
        self._file = None
        with h5py.File(cache_path, "r") as f:
            self.labels = f["labels"][self.indices]

    def __len__(self):
        return len(self.indices)

    def _f(self):
        if self._file is None:
            self._file = h5py.File(self.cache_path, "r")
        return self._file

    def __getitem__(self, idx):
        f = self._f()
        ri = self.indices[idx]
        x = torch.from_numpy(f["signals"][ri]).float()
        if x.size(0) == 5000 and x.size(1) == 12:
            x = x.transpose(0, 1)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

pin = device.type == "cuda"
train_loader = DataLoader(CachedChagasDataset(cache_file, train_indices),
                          batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=pin, drop_last=True)
val_loader   = DataLoader(CachedChagasDataset(cache_file, val_indices),
                          batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=pin)
test_loader  = DataLoader(CachedChagasDataset(cache_file, test_indices),
                          batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=pin)
samitrop_loader = DataLoader(CachedChagasDataset(cache_file, samitrop_indices),
                             batch_size=CFG["batch_size"], shuffle=False,
                             num_workers=CFG["num_workers"], pin_memory=pin)

Cache — Samples: 49,152 | Chagas+ 16.7%
CODE-15% entries : 47,521  (13.8% Chagas+)
SaMi-Trop entries: 1,631  (100% Chagas+, OOD only)

  DATA SPLIT SUMMARY
  Train : 33,121  (13.8% Chagas+)
  Val   :  7,271  (13.8% Chagas+)
  Test  :  7,129  (13.8% Chagas+) [CODE-15% held-out]
  OOD   :  1,631  (100.0% Chagas+) [SaMi-Trop domain-shift]


In [5]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        at = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (at * (1 - pt).pow(self.gamma) * bce).mean()

@dataclass
class EvalResult:
    auroc: float
    auprc: float
    f1: float
    sens: float
    spec: float

def compute_metrics(y_true, y_prob, thr=0.5):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    if len(np.unique(y_true)) < 2:
        return EvalResult(float("nan"), float("nan"), 0.0, 0.0, 0.0)
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    yp = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yp, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    f1 = f1_score(y_true, yp, zero_division=0)
    return EvalResult(auroc, auprc, f1, sens, spec)

In [6]:
class DeepECG_SSL_Classifier(nn.Module):
    def __init__(self, ssl_model_path):
        super().__init__()
        logger.info(f"Loading DeepECG-SSL from {ssl_model_path}")
        model, cfg, task = checkpoint_utils.load_model_and_task(
            ssl_model_path,
            suffix=""
        )
        self.encoder = model
        
        self.head = nn.Sequential(
            nn.Linear(768, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        
    def train(self, mode=True):
        super().train(mode)
        if not next(self.encoder.parameters()).requires_grad:
            self.encoder.eval()
        return self

    def forward(self, x):
        out = self.encoder(source=x, padding_mask=None, features_only=True)
        features = out["x"]
        if features.size(0) != x.size(0) and features.size(1) == x.size(0):
            features = features.transpose(0, 1)
        
        pooled = features.mean(dim=1)
        
        logits = self.head(pooled).view(-1)
        return logits

model = DeepECG_SSL_Classifier(CFG["ssl_model_path"]).to(device)

18:05:32 [INFO] Loading DeepECG-SSL from DeepECG_Docker-main/ssl_pretrained_model/SSL_pretrained.pt
18:05:34 [INFO] Loaded a checkpoint in 1.67s


In [7]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    total_loss = 0
    criterion = FocalLoss(alpha=CFG["focal_alpha"], gamma=CFG["focal_gamma"]).to(device)
    
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item()
        
        probs = torch.sigmoid(logits)
        all_preds.extend(probs.cpu().numpy())
        all_targets.extend(y.cpu().numpy())
        
    metrics = compute_metrics(all_targets, all_preds)
    return total_loss / len(loader), metrics

In [8]:
logger.info("Starting Step A: Linear Probing (freezing encoder)")

LINEAR_CKPT = "approach12_linear_code15_best.pt"

if os.path.exists(LINEAR_CKPT):
    logger.info(f"Found saved linear probing model ({LINEAR_CKPT}). Loading and skipping training.")
    model.load_state_dict(torch.load(LINEAR_CKPT, map_location=device))
    model.eval()
else:
    for param in model.encoder.parameters():
        param.requires_grad = False

    optimizer = AdamW(model.head.parameters(), lr=CFG["lr_linear"], weight_decay=CFG["weight_decay"])
    criterion = FocalLoss(alpha=CFG["focal_alpha"], gamma=CFG["focal_gamma"]).to(device)

    best_val_auroc = 0.0
    patience_counter = 0
    patience = 5

    for epoch in range(CFG["epochs_linear"]):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, metrics = evaluate(model, val_loader, device)

        logger.info(
            f"Epoch {epoch+1}/{CFG['epochs_linear']} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Val AUROC: {metrics.auroc:.4f} | Val AUPRC: {metrics.auprc:.4f}"
        )

        if metrics.auroc > best_val_auroc:
            best_val_auroc = metrics.auroc
            patience_counter = 0
            torch.save(model.state_dict(), LINEAR_CKPT)
            logger.info(f"  -> Saved best linear checkpoint (Val AUROC {best_val_auroc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                logger.info(f"Early stopping at epoch {epoch+1} (patience={patience})")
                break

    model.load_state_dict(torch.load(LINEAR_CKPT, map_location=device))
    logger.info(f"Linear probing done. Best Val AUROC: {best_val_auroc:.4f}")

18:05:34 [INFO] Starting Step A: Linear Probing (freezing encoder)
18:18:14 [INFO] Epoch 1/10 | Train Loss: 0.0429 | Val Loss: 0.0317 | Val AUROC: 0.8124 | Val AUPRC: 0.4516
18:18:15 [INFO]   -> Saved best linear checkpoint (Val AUROC 0.8124)
18:40:46 [INFO] Epoch 2/10 | Train Loss: 0.0326 | Val Loss: 0.0325 | Val AUROC: 0.8139 | Val AUPRC: 0.4541
18:40:47 [INFO]   -> Saved best linear checkpoint (Val AUROC 0.8139)
19:02:19 [INFO] Epoch 3/10 | Train Loss: 0.0323 | Val Loss: 0.0315 | Val AUROC: 0.8147 | Val AUPRC: 0.4473
19:02:20 [INFO]   -> Saved best linear checkpoint (Val AUROC 0.8147)
19:22:58 [INFO] Epoch 4/10 | Train Loss: 0.0325 | Val Loss: 0.0316 | Val AUROC: 0.8183 | Val AUPRC: 0.4567
19:22:59 [INFO]   -> Saved best linear checkpoint (Val AUROC 0.8183)
19:43:26 [INFO] Epoch 5/10 | Train Loss: 0.0323 | Val Loss: 0.0315 | Val AUROC: 0.8194 | Val AUPRC: 0.4515
19:43:27 [INFO]   -> Saved best linear checkpoint (Val AUROC 0.8194)
20:03:55 [INFO] Epoch 6/10 | Train Loss: 0.0322 | Val

In [9]:
logger.info("Starting Step B: Full Fine-Tuning (unfreezing encoder)")

LINEAR_CKPT  = "approach12_linear_code15_best.pt"
FINETUNE_CKPT = "approach12_finetune_code15_best.pt"

if os.path.exists(FINETUNE_CKPT):
    logger.info(f"Found saved fine-tuned model ({FINETUNE_CKPT}). Loading and skipping training.")
    model.load_state_dict(torch.load(FINETUNE_CKPT, map_location=device))
    model.eval()
else:
    model.load_state_dict(torch.load(LINEAR_CKPT, map_location=device))

    for param in model.encoder.parameters():
        param.requires_grad = True

    optimizer = AdamW(model.parameters(), lr=CFG["lr_finetune"], weight_decay=CFG["weight_decay"])
    criterion = FocalLoss(alpha=CFG["focal_alpha"], gamma=CFG["focal_gamma"]).to(device)

    best_val_auroc = 0.0
    patience_counter = 0
    patience = 5

    for epoch in range(CFG["epochs_finetune"]):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, metrics = evaluate(model, val_loader, device)

        logger.info(
            f"Epoch {epoch+1}/{CFG['epochs_finetune']} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Val AUROC: {metrics.auroc:.4f} | Val AUPRC: {metrics.auprc:.4f}"
        )

        if metrics.auroc > best_val_auroc:
            best_val_auroc = metrics.auroc
            patience_counter = 0
            torch.save(model.state_dict(), FINETUNE_CKPT)
            logger.info(f"  -> Saved best fine-tune checkpoint (Val AUROC {best_val_auroc:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                logger.info(f"Early stopping at epoch {epoch+1} (patience={patience})")
                break

    model.load_state_dict(torch.load(FINETUNE_CKPT, map_location=device))
    logger.info(f"Fine-tuning done. Best Val AUROC: {best_val_auroc:.4f}")

21:22:23 [INFO] Starting Step B: Full Fine-Tuning (unfreezing encoder)
22:19:39 [INFO] Epoch 1/20 | Train Loss: 0.0312 | Val Loss: 0.0309 | Val AUROC: 0.8447 | Val AUPRC: 0.5102
22:19:40 [INFO]   -> Saved best fine-tune checkpoint (Val AUROC 0.8447)
23:14:32 [INFO] Epoch 2/20 | Train Loss: 0.0297 | Val Loss: 0.0291 | Val AUROC: 0.8492 | Val AUPRC: 0.5245
23:14:33 [INFO]   -> Saved best fine-tune checkpoint (Val AUROC 0.8492)
00:11:13 [INFO] Epoch 3/20 | Train Loss: 0.0288 | Val Loss: 0.0311 | Val AUROC: 0.8427 | Val AUPRC: 0.4768
01:01:11 [INFO] Epoch 4/20 | Train Loss: 0.0282 | Val Loss: 0.0292 | Val AUROC: 0.8582 | Val AUPRC: 0.5484
01:01:12 [INFO]   -> Saved best fine-tune checkpoint (Val AUROC 0.8582)
01:28:22 [INFO] Epoch 5/20 | Train Loss: 0.0270 | Val Loss: 0.0292 | Val AUROC: 0.8560 | Val AUPRC: 0.5398
01:54:20 [INFO] Epoch 6/20 | Train Loss: 0.0260 | Val Loss: 0.0296 | Val AUROC: 0.8473 | Val AUPRC: 0.5236
02:21:09 [INFO] Epoch 7/20 | Train Loss: 0.0251 | Val Loss: 0.0296 | Va

In [10]:

FINETUNE_CKPT = "approach12_finetune_code15_best.pt"
model.load_state_dict(torch.load(FINETUNE_CKPT, map_location=device))
model.eval()

test_loss, test_metrics = evaluate(model, test_loader, device)

print()
print(f"{'='*62}")
print(f"  CODE-15% TEST SET EVALUATION (n={len(test_indices):,})")
print(f"  Positive rate: {100*all_labels[test_indices].mean():.1f}%")
print(f"{'='*62}")
print(f"  AUROC        : {test_metrics.auroc:.4f}")
print(f"  AUPRC        : {test_metrics.auprc:.4f}")
print(f"  F1 (thr=0.5) : {test_metrics.f1:.4f}")
print(f"  Sensitivity  : {test_metrics.sens:.4f}")
print(f"  Specificity  : {test_metrics.spec:.4f}")
print(f"  Focal loss   : {test_loss:.4f}")
print(f"{'='*62}")

logger.info(
    f"CODE-15% Test | AUROC: {test_metrics.auroc:.4f} | "
    f"AUPRC: {test_metrics.auprc:.4f} | F1: {test_metrics.f1:.4f}"
)

03:15:00 [INFO] CODE-15% Test | AUROC: 0.8593 | AUPRC: 0.5813 | F1: 0.2818



  CODE-15% TEST SET EVALUATION (n=7,129)
  Positive rate: 13.8%
  AUROC        : 0.8593
  AUPRC        : 0.5813
  F1 (thr=0.5) : 0.2818
  Sensitivity  : 0.1677
  Specificity  : 0.9964
  Focal loss   : 0.0285


In [11]:
LAST_CKPT = "approach12_finetune_code15_last.pt"
torch.save(model.state_dict(), LAST_CKPT)
logger.info(f"Saved {LAST_CKPT} (state_dict only)")

03:15:01 [INFO] Saved approach12_finetune_code15_last.pt (state_dict only)


## OOD evaluation: SaMi-Trop

SaMi-Trop (n=1,631, all Chagas+) comes from a different hospital and device than CODE-15% and was never used in train/val/test. Since it has only one class, AUROC is undefined on it alone, so we report two things:

1. Recall on SaMi-Trop at threshold 0.5.
2. AUROC on SaMi-Trop positives combined with the held-out CODE-15% test negatives.

In [12]:

import warnings

FINETUNE_CKPT = "approach12_finetune_code15_best.pt"
assert Path(FINETUNE_CKPT).exists(), (
    f"Checkpoint {FINETUNE_CKPT} not found. "
    "Run the fine-tuning cell first."
)

ood_model = DeepECG_SSL_Classifier(CFG["ssl_model_path"]).to(device)
ood_model.load_state_dict(torch.load(FINETUNE_CKPT, map_location=device))
ood_model.eval()
logger.info(f"Loaded fine-tuned weights from '{FINETUNE_CKPT}'")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _, samitrop_metrics = evaluate(ood_model, samitrop_loader, device)

print()
print(f"{'='*62}")
print(f"  OOD EVALUATION — SaMi-Trop (domain-shift)")
print(f"  n={len(samitrop_indices):,}  |  100% Chagas+  |  different hospital/device")
print(f"{'='*62}")
print(f"  Sensitivity (thr=0.5): {samitrop_metrics.sens:.4f}")
print(f"  F1 (thr=0.5)         : {samitrop_metrics.f1:.4f}")
print(f"  AUROC                : N/A (single class)")
print(f"{'='*62}")

test_neg_mask    = all_labels[test_indices] == 0
test_neg_indices = test_indices[test_neg_mask]

combined_indices = np.concatenate([samitrop_indices, test_neg_indices])
n_pos = len(samitrop_indices)
n_neg = len(test_neg_indices)
prevalence = n_pos / (n_pos + n_neg)
logger.info(
    f"Combined OOD set | n={len(combined_indices):,} | "
    f"Chagas+ {n_pos:,} ({100*prevalence:.1f}%) | "
    f"Chagas- {n_neg:,} ({100*(1-prevalence):.1f}%)"
)

combined_loader = DataLoader(
    CachedChagasDataset(cache_file, combined_indices),
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=(device.type == "cuda"),
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ood_loss, ood_metrics = evaluate(ood_model, combined_loader, device)

print()
print(f"{'='*62}")
print(f"  COMBINED OOD AUROC")
print(f"  SaMi-Trop positives + CODE-15% test negatives")
print(f"  n={len(combined_indices):,}  |  Chagas+ {n_pos:,}  |  Chagas- {n_neg:,}")
print(f"{'='*62}")
print(f"  AUROC        : {ood_metrics.auroc:.4f}")
print(f"  AUPRC        : {ood_metrics.auprc:.4f}")
print(f"  F1 (thr=0.5) : {ood_metrics.f1:.4f}")
print(f"  Sensitivity  : {ood_metrics.sens:.4f}")
print(f"  Specificity  : {ood_metrics.spec:.4f}")
print(f"  Focal loss   : {ood_loss:.4f}")
print(f"{'='*62}")
print()
print("Negatives: CODE-15% test set (withheld from training and validation).")
print("Positives: SaMi-Trop (different hospital/device — true domain shift).")

03:15:01 [INFO] Loading DeepECG-SSL from DeepECG_Docker-main/ssl_pretrained_model/SSL_pretrained.pt
03:15:03 [INFO] Loaded a checkpoint in 2.63s
03:15:10 [INFO] Loaded fine-tuned weights from 'approach12_finetune_code15_best.pt'
03:15:34 [INFO] Combined OOD set | n=7,776 | Chagas+ 1,631 (21.0%) | Chagas- 6,145 (79.0%)



  OOD EVALUATION — SaMi-Trop (domain-shift)
  n=1,631  |  100% Chagas+  |  different hospital/device
  Sensitivity (thr=0.5): 0.0000
  F1 (thr=0.5)         : 0.0000
  AUROC                : N/A (single class)



  COMBINED OOD AUROC
  SaMi-Trop positives + CODE-15% test negatives
  n=7,776  |  Chagas+ 1,631  |  Chagas- 6,145
  AUROC        : 0.6866
  AUPRC        : 0.4269
  F1 (thr=0.5) : 0.1107
  Sensitivity  : 0.0595
  Specificity  : 0.9961
  Focal loss   : 0.0482

Negatives: CODE-15% test set (withheld from training and validation).
Positives: SaMi-Trop (different hospital/device — true domain shift).
